# TMJ Heatmap Detector — DataSphere Training

**Self-contained notebook** — no repo import required.

Pipeline:
1. Install deps, set paths
2. Download & extract pre-processed volumes (~63 MB) + annotations (~1 MB)
3. Define model, dataset, loss inline
4. Train 3D U-Net, save best checkpoint to filestore

## 1. Setup

In [ ]:
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'tqdm', 'scipy'])
print('deps ok')

In [ ]:
import os, sys, json, random, logging, datetime
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from scipy import ndimage
from tqdm.auto import tqdm

# ── Paths ──────────────────────────────────────────────────────────────────
FILESTORE    = Path('/home/jupyter/filestore')          # persistent storage
DATA_DIR     = FILESTORE / 'heatmap_data'
VOLUMES_DIR  = DATA_DIR / 'heatmap_volumes'             # .npy uint8 volumes
ANN_DIR      = DATA_DIR / 'roi_annotations'             # *_rois.json
SPLIT_JSON   = DATA_DIR / 'detector_split.json'
EXP_DIR      = FILESTORE / 'experiments'

DATA_DIR.mkdir(parents=True, exist_ok=True)
EXP_DIR.mkdir(parents=True, exist_ok=True)

# ── Device ─────────────────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'PyTorch: {torch.__version__}')

# numpy compat fix for PyTorch 2.0.x + numpy 2.x
if not hasattr(np, 'core'):
    import numpy.core
    sys.modules.setdefault('numpy._core', numpy.core)
elif not hasattr(np.core, 'multiarray'):
    sys.modules.setdefault('numpy._core', np.core)

## 2. Download data

In [ ]:
import urllib.request, tarfile

RELEASE_BASE = 'https://github.com/tzopiz/MasterProject/releases/download'

def download_release_asset(tag: str, filename: str, dest: Path):
    if dest.exists():
        print(f'  Already exists: {dest}')
        return
    url = f'{RELEASE_BASE}/{tag}/{filename}'
    print(f'  Downloading {url} ...')
    tmp = dest.parent / (dest.name + '.tmp')
    urllib.request.urlretrieve(url, tmp)
    tmp.rename(dest)
    print(f'  Saved → {dest} ({dest.stat().st_size/1e6:.1f} MB)')


# ── Volumes (~63 MB) ───────────────────────────────────────────────────────
vol_tar = DATA_DIR / 'heatmap_volumes.tar.gz'
download_release_asset('heatmap-volumes-v1', 'heatmap_volumes.tar.gz', vol_tar)

if not VOLUMES_DIR.exists() or not any(VOLUMES_DIR.glob('*.npy')):
    print('Extracting volumes...')
    with tarfile.open(vol_tar) as tf:
        tf.extractall(DATA_DIR)
    npy_count = len(list(VOLUMES_DIR.glob('*.npy')))
    print(f'Extracted {npy_count} volumes → {VOLUMES_DIR}')
else:
    print(f'Volumes already extracted: {len(list(VOLUMES_DIR.glob("*.npy")))} files')

# ── Annotations (~200 KB) ─────────────────────────────────────────────────
ann_tar = DATA_DIR / 'roi_annotations.tar.gz'
download_release_asset('annotations-v1', 'roi_annotations.tar.gz', ann_tar)

if not ANN_DIR.exists() or not any(ANN_DIR.glob('*.json')):
    print('Extracting annotations...')
    with tarfile.open(ann_tar) as tf:
        tf.extractall(DATA_DIR)
    n = len(list(ANN_DIR.glob('*.json')))
    print(f'Extracted {n} annotations → {ANN_DIR}')
else:
    print(f'Annotations already present: {len(list(ANN_DIR.glob("*.json")))} files')

# ── Split JSON ─────────────────────────────────────────────────────────────
split_src = DATA_DIR / 'detector_split.json'
download_release_asset('annotations-v1', 'detector_split.json', split_src)
if not SPLIT_JSON.exists():
    split_src.rename(SPLIT_JSON)

with open(SPLIT_JSON) as f:
    split = json.load(f)
print(f"Split — train:{len(split['train'])}  val:{len(split['val'])}  test:{len(split['test'])}")

## 3. Model

In [ ]:
from typing import List, Optional, Tuple


def _double_conv(in_ch: int, out_ch: int) -> nn.Sequential:
    return nn.Sequential(
        nn.Conv3d(in_ch, out_ch, 3, padding=1, bias=False),
        nn.BatchNorm3d(out_ch), nn.ReLU(inplace=True),
        nn.Conv3d(out_ch, out_ch, 3, padding=1, bias=False),
        nn.BatchNorm3d(out_ch), nn.ReLU(inplace=True),
    )


class _EncoderBlock(nn.Module):
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        self.conv = _double_conv(in_ch, out_ch)
        self.pool = nn.MaxPool3d(2)

    def forward(self, x):
        skip = self.conv(x)
        return self.pool(skip), skip


class _DecoderBlock(nn.Module):
    def __init__(self, in_ch: int, skip_ch: int, out_ch: int):
        super().__init__()
        self.up   = nn.ConvTranspose3d(in_ch, in_ch // 2, kernel_size=2, stride=2)
        self.conv = _double_conv(in_ch // 2 + skip_ch, out_ch)

    def forward(self, x, skip):
        x = self.up(x)
        if x.shape != skip.shape:
            x = F.pad(x, [0, skip.shape[4]-x.shape[4],
                           0, skip.shape[3]-x.shape[3],
                           0, skip.shape[2]-x.shape[2]])
        return self.conv(torch.cat([skip, x], dim=1))


class TMJHeatmapDetector(nn.Module):
    """3D U-Net: (B,1,D,H,W) → (B,1,D,H,W) raw logits. One joint per model."""

    def __init__(self, in_channels: int = 1,
                 features: Optional[List[int]] = None):
        super().__init__()
        if features is None:
            features = [32, 64, 128, 256]

        self.encoders = nn.ModuleList()
        prev = in_channels
        for f in features:
            self.encoders.append(_EncoderBlock(prev, f))
            prev = f

        self.bottleneck = _double_conv(features[-1], features[-1] * 2)
        prev = features[-1] * 2

        self.decoders = nn.ModuleList()
        for f in reversed(features):
            self.decoders.append(_DecoderBlock(prev, f, f))
            prev = f

        self.head = nn.Conv3d(features[0], 1, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        skips = []
        for enc in self.encoders:
            x, skip = enc(x)
            skips.append(skip)
        x = self.bottleneck(x)
        for dec, skip in zip(self.decoders, reversed(skips)):
            x = dec(x, skip)
        return self.head(x)   # (B, 1, D, H, W)


# Два независимых детектора
left_model  = TMJHeatmapDetector().to(device)
right_model = TMJHeatmapDetector().to(device)
n = sum(p.numel() for p in left_model.parameters())
print(f'Parameters per model: {n/1e6:.2f}M  (×2 total)')

## 4. Dataset

In [ ]:
# ── Heatmap utilities ──────────────────────────────────────────────────────

TARGET_SHAPE = (96, 128, 128)

def make_heatmap_scaled(shape, center_zyx, sigma, scale):
    D, H, W = shape
    cz = center_zyx[0] * scale[0]
    cy = center_zyx[1] * scale[1]
    cx = center_zyx[2] * scale[2]
    z = np.arange(D, dtype=np.float32)
    y = np.arange(H, dtype=np.float32)
    x = np.arange(W, dtype=np.float32)
    dist_sq = (
        (z[:, None, None] - cz) ** 2
        + (y[None, :, None] - cy) ** 2
        + (x[None, None, :] - cx) ** 2
    )
    return np.exp(-dist_sq / (2.0 * sigma ** 2)).astype(np.float32)


# ── Augmentation ───────────────────────────────────────────────────────────

def _augment_single(volume, center_orig, orig_shape):
    """Augment volume + single joint center. Returns (vol, new_center)."""
    if random.random() < 0.7:
        volume = np.clip(random.uniform(0.9, 1.1) * volume + random.uniform(-0.05, 0.05), 0.0, 1.0)
    if random.random() < 0.4:
        volume = np.clip(volume + np.random.normal(0, 0.01, volume.shape), 0.0, 1.0).astype(np.float32)
    # No x-flip here — would swap L/R which doesn't make sense for single-joint model
    if random.random() < 0.5:
        angle = random.uniform(-10, 10)
        axes  = random.choice([(0, 1), (0, 2), (1, 2)])
        volume = ndimage.rotate(volume, angle, axes=axes, reshape=False, order=1, mode='nearest')
        D, H, W = volume.shape
        centers = [D / 2, H / 2, W / 2]
        a, b = axes
        c = np.cos(np.radians(angle)); s = np.sin(np.radians(angle))
        orig = list(center_orig)
        scale_ds = [TARGET_SHAPE[i] / orig_shape[i] for i in range(3)]
        da = orig[a] * scale_ds[a] - centers[a]
        db = orig[b] * scale_ds[b] - centers[b]
        new_a_ds = da * c - db * s + centers[a]
        new_b_ds = da * s + db * c + centers[b]
        orig[a] = int(np.clip(new_a_ds / scale_ds[a], 0, orig_shape[a] - 1))
        orig[b] = int(np.clip(new_b_ds / scale_ds[b], 0, orig_shape[b] - 1))
        center_orig = orig
    return volume.astype(np.float32), center_orig


# ── Single-joint Dataset ───────────────────────────────────────────────────

class TMJSingleJointDataset(Dataset):
    """One heatmap per sample — either left OR right joint."""

    def __init__(self, study_ids, annotations_dir, volumes_dir,
                 side='left', sigma=6.0, is_train=True):
        assert side in ('left', 'right')
        self.side      = side
        self.sigma     = sigma
        self.is_train  = is_train
        self.ann_dir   = Path(annotations_dir)
        self.vol_dir   = Path(volumes_dir)

        self.records = []
        key = 'left_tmj' if side == 'left' else 'right_tmj'
        for sid in study_ids:
            ann_path = self.ann_dir / f'{sid}_rois.json'
            if not ann_path.exists():
                continue
            with open(ann_path) as f:
                ann = json.load(f)
            self.records.append({
                'study_id':     ann['scan_id'],
                'center':       list(ann[key]['center']),
                'orig_shape':   ann['original_shape'],
            })
        print(f'TMJSingleJointDataset [{side}]: {len(self.records)} samples')

    def _load_volume(self, sid):
        npy = self.vol_dir / f'{sid}.npy'
        vol = np.array(np.load(str(npy)), dtype=np.float32) / 255.0
        if vol.shape != TARGET_SHAPE:
            zoom = [t / s for t, s in zip(TARGET_SHAPE, vol.shape)]
            vol = ndimage.zoom(vol, zoom, order=1)
        return vol.astype(np.float32)

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec    = self.records[idx]
        vol    = self._load_volume(rec['study_id'])
        center = list(rec['center'])
        orig   = rec['orig_shape']

        if self.is_train:
            vol, center = _augment_single(vol, center, orig)

        scale = [TARGET_SHAPE[i] / orig[i] for i in range(3)]
        hm    = make_heatmap_scaled(TARGET_SHAPE, center, self.sigma, scale)

        vol_t = torch.tensor(vol, dtype=torch.float32).unsqueeze(0)   # (1,D,H,W)
        hm_t  = torch.tensor(hm,  dtype=torch.float32).unsqueeze(0)   # (1,D,H,W)
        return vol_t, hm_t


# ── Loss ───────────────────────────────────────────────────────────────────

def weighted_mse_loss(pred, target, pos_weight=100.0):
    weight = 1.0 + pos_weight * target
    return (weight * (pred - target) ** 2).mean()


print('Utilities defined')

In [ ]:
# ── Hyperparams ────────────────────────────────────────────────────────────
SIGMA       = 6.0
BATCH_SIZE  = 4
NUM_WORKERS = 0
LR          = 1e-4
WEIGHT_DECAY= 1e-4
POS_WEIGHT  = 100.0
EPOCHS      = 200
LR_PATIENCE = 15
EARLY_STOP  = 40

if device.type == 'cuda':
    torch.backends.cudnn.benchmark = True

# Два датасета — левый и правый сустав отдельно
left_train_ds  = TMJSingleJointDataset(split['train'], ANN_DIR, VOLUMES_DIR, side='left',  sigma=SIGMA, is_train=True)
left_val_ds    = TMJSingleJointDataset(split['val'],   ANN_DIR, VOLUMES_DIR, side='left',  sigma=SIGMA, is_train=False)
right_train_ds = TMJSingleJointDataset(split['train'], ANN_DIR, VOLUMES_DIR, side='right', sigma=SIGMA, is_train=True)
right_val_ds   = TMJSingleJointDataset(split['val'],   ANN_DIR, VOLUMES_DIR, side='right', sigma=SIGMA, is_train=False)

left_train_loader  = DataLoader(left_train_ds,  batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS)
left_val_loader    = DataLoader(left_val_ds,    batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
right_train_loader = DataLoader(right_train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS)
right_val_loader   = DataLoader(right_val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print(f'Left  — train: {len(left_train_loader)} batches  val: {len(left_val_loader)} batches')
print(f'Right — train: {len(right_train_loader)} batches  val: {len(right_val_loader)} batches')

## 5. Training

In [ ]:
def argmax_coords_t(heatmap: torch.Tensor):
    """Peak via argmax. (D,H,W) → [z,y,x]."""
    idx = int(heatmap.argmax())
    D, H, W = heatmap.shape
    return torch.tensor([idx//(H*W), (idx%(H*W))//W, idx%W], dtype=torch.float32)


def compute_mae_batch(pred_hm, target_hm):
    """MAE in downsampled px. pred/target: (B,1,D,H,W) CPU float32."""
    pred_hm, target_hm = pred_hm.float(), target_hm.float()
    errs = []
    for b in range(pred_hm.shape[0]):
        pc = argmax_coords_t(torch.sigmoid(pred_hm[b, 0]))
        tc = argmax_coords_t(target_hm[b, 0])
        errs.append(torch.sqrt(((pc - tc) ** 2).sum()).item())
    return float(np.mean(errs))


def run_epoch(model, loader, optimizer, scaler, is_train, epoch_num, tag):
    model.train() if is_train else model.eval()
    total_loss, all_mae = 0.0, []
    ctx = torch.enable_grad() if is_train else torch.no_grad()
    with ctx:
        for vols, targets in tqdm(loader, desc=f'[{epoch_num}] {tag}', leave=False):
            vols, targets = vols.to(device), targets.to(device)
            if is_train:
                optimizer.zero_grad()
                if scaler:
                    with torch.cuda.amp.autocast():
                        pred = model(vols)
                        loss = weighted_mse_loss(torch.sigmoid(pred), targets, POS_WEIGHT)
                    scaler.scale(loss).backward()
                    scaler.step(optimizer); scaler.update()
                else:
                    pred = model(vols)
                    loss = weighted_mse_loss(torch.sigmoid(pred), targets, POS_WEIGHT)
                    loss.backward(); optimizer.step()
            else:
                pred = model(vols)
                loss = weighted_mse_loss(torch.sigmoid(pred), targets, POS_WEIGHT)
                all_mae.append(compute_mae_batch(pred.detach().cpu(), targets.cpu()))
            total_loss += loss.item()
    result = {'loss': total_loss / len(loader)}
    if all_mae:
        result['mae'] = float(np.mean(all_mae))
    return result


def train_model(model, train_loader, val_loader, side):
    ts      = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
    exp_dir = EXP_DIR / f'heatmap_{side}_{ts}'
    exp_dir.mkdir(parents=True, exist_ok=True)

    config = dict(side=side, sigma=SIGMA, batch_size=BATCH_SIZE, lr=LR,
                  pos_weight=POS_WEIGHT, epochs=EPOCHS, heatmap=True, timestamp=ts)
    with open(exp_dir / 'config.json', 'w') as f:
        json.dump(config, f, indent=2)

    optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min',
                                                      factor=0.5, patience=LR_PATIENCE)
    scaler = torch.cuda.amp.GradScaler() if device.type == 'cuda' else None

    best_mae, no_imp = float('inf'), 0

    print(f'\n=== {side.upper()} JOINT ===')
    print(f'{"Ep":>4}  {"tr_loss":>8}  │  {"va_loss":>8}  {"va_mae":>7}')
    print('─' * 42)

    for epoch in range(1, EPOCHS + 1):
        tr  = run_epoch(model, train_loader, optimizer, scaler, True,  epoch, f'{side[:1].upper()} Train')
        val = run_epoch(model, val_loader,   optimizer, scaler, False, epoch, f'{side[:1].upper()} Val  ')
        scheduler.step(val['mae'])
        lr_now = optimizer.param_groups[0]['lr']

        line = (f"{epoch:4d}  {tr['loss']:8.4f}  │  "
                f"{val['loss']:8.4f}  {val['mae']:7.2f}  lr={lr_now:.1e}")

        if val['mae'] < best_mae:
            best_mae = val['mae']
            no_imp   = 0
            torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(),
                        'best_val_mae': best_mae, 'config': config},
                       exp_dir / 'best_model.pth')
            line += f'  ✓ ({best_mae:.2f}ds ≈ {best_mae*6*0.4:.1f}mm)'
        else:
            no_imp += 1

        print(line)
        with open(exp_dir / 'metrics.jsonl', 'a') as f:
            f.write(json.dumps({'epoch': epoch, 'lr': lr_now,
                                'train_loss': tr['loss'],
                                'val_loss': val['loss'], 'val_mae': val['mae']}) + '\n')

        if EARLY_STOP > 0 and no_imp >= EARLY_STOP:
            print(f'Early stopping at epoch {epoch}')
            break

    print(f'Best {side} MAE: {best_mae:.2f} ds_px ≈ {best_mae*6*0.4:.1f} mm')
    return exp_dir, best_mae


print('Training functions ready')

In [ ]:
ts = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
left_exp_dir  = EXP_DIR / f'heatmap_left_{ts}'
right_exp_dir = EXP_DIR / f'heatmap_right_{ts}'
left_exp_dir.mkdir(parents=True, exist_ok=True)
right_exp_dir.mkdir(parents=True, exist_ok=True)

opt_l  = optim.Adam(left_model.parameters(),  lr=LR, weight_decay=WEIGHT_DECAY)
opt_r  = optim.Adam(right_model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
sched_l = optim.lr_scheduler.ReduceLROnPlateau(opt_l, mode='min', factor=0.5, patience=LR_PATIENCE)
sched_r = optim.lr_scheduler.ReduceLROnPlateau(opt_r, mode='min', factor=0.5, patience=LR_PATIENCE)
scaler_l = torch.cuda.amp.GradScaler() if device.type == 'cuda' else None
scaler_r = torch.cuda.amp.GradScaler() if device.type == 'cuda' else None

best = {'left': float('inf'), 'right': float('inf')}
no_imp = {'left': 0, 'right': 0}
done = {'left': False, 'right': False}

print(f'{"Ep":>4}  {"L_tr":>8}  {"L_val":>8}  {"L_mae":>7}  │  {"R_tr":>8}  {"R_val":>8}  {"R_mae":>7}')
print('─' * 70)

for epoch in range(1, EPOCHS + 1):
    if done['left'] and done['right']:
        break

    # ── Train pass — оба батч-лоадера зиппятся ─────────────────────────────
    left_model.train(); right_model.train()
    tl_loss = tr_loss = 0.0

    for (lv, lt), (rv, rt) in zip(left_train_loader, right_train_loader):
        lv, lt = lv.to(device), lt.to(device)
        rv, rt = rv.to(device), rt.to(device)

        if not done['left']:
            opt_l.zero_grad()
            if scaler_l:
                with torch.cuda.amp.autocast():
                    lp = left_model(lv)
                    ll = weighted_mse_loss(torch.sigmoid(lp), lt, POS_WEIGHT)
                scaler_l.scale(ll).backward(); scaler_l.step(opt_l); scaler_l.update()
            else:
                lp = left_model(lv); ll = weighted_mse_loss(torch.sigmoid(lp), lt, POS_WEIGHT)
                ll.backward(); opt_l.step()
            tl_loss += ll.item()

        if not done['right']:
            opt_r.zero_grad()
            if scaler_r:
                with torch.cuda.amp.autocast():
                    rp = right_model(rv)
                    rl = weighted_mse_loss(torch.sigmoid(rp), rt, POS_WEIGHT)
                scaler_r.scale(rl).backward(); scaler_r.step(opt_r); scaler_r.update()
            else:
                rp = right_model(rv); rl = weighted_mse_loss(torch.sigmoid(rp), rt, POS_WEIGHT)
                rl.backward(); opt_r.step()
            tr_loss += rl.item()

    n_tr = len(left_train_loader)
    tl_loss /= n_tr; tr_loss /= n_tr

    # ── Val pass ────────────────────────────────────────────────────────────
    left_model.eval(); right_model.eval()
    vl_loss = vl_mae = vr_loss = vr_mae = 0.0

    with torch.no_grad():
        for (lv, lt), (rv, rt) in zip(left_val_loader, right_val_loader):
            lv, lt = lv.to(device), lt.to(device)
            rv, rt = rv.to(device), rt.to(device)

            lp = left_model(lv);  ll = weighted_mse_loss(torch.sigmoid(lp), lt, POS_WEIGHT)
            rp = right_model(rv); rl = weighted_mse_loss(torch.sigmoid(rp), rt, POS_WEIGHT)
            vl_loss += ll.item(); vr_loss += rl.item()
            vl_mae  += compute_mae_batch(lp.cpu(), lt.cpu())
            vr_mae  += compute_mae_batch(rp.cpu(), rt.cpu())

    n_val = len(left_val_loader)
    vl_loss /= n_val; vl_mae /= n_val
    vr_loss /= n_val; vr_mae /= n_val

    sched_l.step(vl_mae); sched_r.step(vr_mae)
    lr_l = opt_l.param_groups[0]['lr']; lr_r = opt_r.param_groups[0]['lr']

    line = (f"{epoch:4d}  {tl_loss:8.4f}  {vl_loss:8.4f}  {vl_mae:7.2f}  │  "
            f"{tr_loss:8.4f}  {vr_loss:8.4f}  {vr_mae:7.2f}")

    for side, mae, exp_dir, model_, best_key in [
        ('left',  vl_mae, left_exp_dir,  left_model,  'left'),
        ('right', vr_mae, right_exp_dir, right_model, 'right'),
    ]:
        if mae < best[best_key]:
            best[best_key] = mae; no_imp[best_key] = 0
            torch.save({'epoch': epoch, 'model_state_dict': model_.state_dict(),
                        'best_val_mae': mae}, exp_dir / 'best_model.pth')
            line += f'  ✓{side[0].upper()}({mae:.1f}ds≈{mae*6*0.4:.0f}mm)'
        else:
            no_imp[best_key] += 1
            if EARLY_STOP > 0 and no_imp[best_key] >= EARLY_STOP and not done[best_key]:
                done[best_key] = True
                line += f'  STOP_{side[0].upper()}'

    print(line)
    for side, metrics in [('left', {'epoch': epoch, 'lr': lr_l, 'train_loss': tl_loss, 'val_loss': vl_loss, 'val_mae': vl_mae}),
                           ('right', {'epoch': epoch, 'lr': lr_r, 'train_loss': tr_loss, 'val_loss': vr_loss, 'val_mae': vr_mae})]:
        exp = left_exp_dir if side == 'left' else right_exp_dir
        with open(exp / 'metrics.jsonl', 'a') as f:
            f.write(json.dumps(metrics) + '\n')

print(f'\nLEFT  best: {best["left"]:.2f} ds_px ≈ {best["left"]*6*0.4:.1f} mm')
print(f'RIGHT best: {best["right"]:.2f} ds_px ≈ {best["right"]*6*0.4:.1f} mm')

## 6. Upload checkpoint to GitHub Release (optional)

## 6. Визуализация: GT vs Predicted на трейне

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

N_SAMPLES = 6  # сколько примеров показать

# ── Берём первые N_SAMPLES из train без аугментации ────────────────────────
vis_ds = TMJHeatmapDataset(split['train'][:N_SAMPLES], ANN_DIR, VOLUMES_DIR,
                            sigma=SIGMA, downsample_factor=DOWNSAMPLE_FACTOR,
                            is_train=False)  # is_train=False → без аугментации

model.eval()
fig, axes = plt.subplots(N_SAMPLES, 3, figsize=(13, 4 * N_SAMPLES))
fig.suptitle('GT (зелёный) vs Predicted (красный)\nAxial | Coronal | Sagittal',
             fontsize=13, y=1.01)

with torch.no_grad():
    for i, (vol_t, hm_t) in enumerate(vis_ds):
        study_id = vis_ds.records[i]['study_id']
        vol_np   = vol_t.squeeze(0).numpy()          # (D,H,W)

        # ── GT координаты (downsampled) ────────────────────────────────────
        gt_left,  _ = coords_from_heatmap(hm_t[0], DOWNSAMPLE_FACTOR)
        gt_right, _ = coords_from_heatmap(hm_t[1], DOWNSAMPLE_FACTOR)
        gt_l = gt_left.numpy().astype(int)    # [z,y,x]
        gt_r = gt_right.numpy().astype(int)

        # ── Predicted координаты ────────────────────────────────────────────
        pred_logits = model(vol_t.unsqueeze(0).to(device))   # (1,2,D,H,W)
        pred_sig    = torch.sigmoid(pred_logits).squeeze(0).cpu()  # (2,D,H,W)
        pr_left,  _ = coords_from_heatmap(pred_sig[0].float(), DOWNSAMPLE_FACTOR)
        pr_right, _ = coords_from_heatmap(pred_sig[1].float(), DOWNSAMPLE_FACTOR)
        pr_l = pr_left.numpy().astype(int)
        pr_r = pr_right.numpy().astype(int)

        # Срез по среднему z между левым и правым GT
        mid_z = int((gt_l[0] + gt_r[0]) / 2)
        mid_z = int(np.clip(mid_z, 0, vol_np.shape[0] - 1))

        # ── Три вида ────────────────────────────────────────────────────────
        views = [
            ('Axial',    vol_np[mid_z],                  # (H,W)
             [(gt_l[1], gt_l[2]), (gt_r[1], gt_r[2])],  # GT (y,x)
             [(pr_l[1], pr_l[2]), (pr_r[1], pr_r[2])]), # Pred (y,x)

            ('Coronal',  vol_np[:, gt_l[1], :],          # (D,W) rows=Z cols=X
             [(gt_l[0], gt_l[2]), (gt_r[0], gt_r[2])],
             [(pr_l[0], pr_l[2]), (pr_r[0], pr_r[2])]),

            ('Sagittal', vol_np[:, :, gt_l[2]],          # (D,H) rows=Z cols=Y
             [(gt_l[0], gt_l[1]), (gt_r[0], gt_r[1])],
             [(pr_l[0], pr_l[1]), (pr_r[0], pr_r[1])]),
        ]

        for j, (title, img, gt_pts, pr_pts) in enumerate(views):
            ax = axes[i][j]
            ax.imshow(img, cmap='gray', origin='upper', aspect='auto')
            for (gy, gx), (py, px) in zip(gt_pts, pr_pts):
                ax.plot(gx, gy, 'o', color='lime',   ms=8, mew=1.5, mec='white')
                ax.plot(px, py, 'x', color='red',    ms=8, mew=2)
                ax.plot([gx, px], [gy, py], '--', color='yellow', lw=0.8, alpha=0.7)
            if j == 0:
                ax.set_ylabel(study_id, fontsize=8)
            ax.set_title(f'{title} z={mid_z}' if j == 0 else title, fontsize=8)
            ax.axis('off')

patch_gt   = mpatches.Patch(color='lime', label='GT')
patch_pred = mpatches.Patch(color='red',  label='Predicted')
fig.legend(handles=[patch_gt, patch_pred], loc='upper right', fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# ── Sanity check: может ли модель переобучиться на 1 примере? ──────────────
# Если нет → проблема в архитектуре или loss. Если да → нужно больше данных/эпох.

one_ds = TMJHeatmapDataset(split['train'][:1], ANN_DIR, VOLUMES_DIR,
                            sigma=SIGMA, downsample_factor=DOWNSAMPLE_FACTOR, is_train=False)
vol_t, hm_t = one_ds[0]
vols    = vol_t.unsqueeze(0).to(device)
targets = hm_t.unsqueeze(0).to(device)

# Новая модель с нуля
overfit_model = TMJHeatmapDetector().to(device)
opt = optim.Adam(overfit_model.parameters(), lr=1e-3)

print('Overfit test on 1 sample:')
print(f'{"Ep":>5}  {"loss":>8}  {"MAE_L":>7}  {"MAE_R":>7}')
for ep in range(1, 201):
    overfit_model.train()
    opt.zero_grad()
    pred = overfit_model(vols)
    loss = weighted_mse_loss(torch.sigmoid(pred), targets, POS_WEIGHT)
    loss.backward()
    opt.step()

    if ep % 20 == 0:
        overfit_model.eval()
        with torch.no_grad():
            p = torch.sigmoid(overfit_model(vols)).squeeze(0).cpu()
        for ch, side in enumerate(['L', 'R']):
            pr = argmax_coords_t(p[ch]).numpy().astype(int)
            gt = np.unravel_index(hm_t[ch].numpy().argmax(), hm_t[ch].shape)
            err = np.linalg.norm(np.array(gt) - pr)
            if ch == 0:
                print(f'{ep:5d}  {loss.item():8.4f}  {err:7.2f}', end='')
            else:
                print(f'  {err:7.2f}  GT_L={np.array(np.unravel_index(hm_t[0].argmax().item(), hm_t[0].shape))} PR_L={argmax_coords_t(p[0]).numpy().astype(int)}  GT_R={np.array(gt)} PR_R={pr}')

print('\nЕсли MAE < 5 к ep 200 → архитектура ОК, нужно больше данных/эпох')
print('Если MAE > 20 → проблема в loss или архитектуре')